# Trade-off Especialização vs Generalização em LLMs via Fine-Tuning

**ICC220 / PPGINF528 — 2º Trabalho Prático**

Pipeline completo: baseline → fine-tuning LoRA/QLoRA → avaliação Text-to-SQL (Execution Accuracy) + MMLU → análise de regressão.

> **Runtime recomendado:** GPU T4 (Colab gratuito). Certifique-se de que `Ambiente de execução → Alterar tipo → GPU` está ativo antes de começar.

## 1. Setup do repositório e dependências

In [ ]:
# Clonar o repositório (ajuste a URL para o seu fork)
# !git clone https://github.com/SEU_USUARIO/text2sql_finetuning.git
# %cd text2sql_finetuning

import os
assert os.path.exists("requirements.txt"), "Execute a partir da raiz do repositório"

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os
import sys

os.environ["PYTHONPATH"] = os.getcwd()
sys.path.insert(0, os.getcwd())

os.environ["BASE_MODEL"] = "Qwen/Qwen2.5-3B-Instruct"
os.environ["SEED"] = "42"

import torch
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram_gb:.1f} GB")

## 2. Download e verificação do Spider

Faça o download do arquivo oficial `spider_data` e envie o zip para a sessão, ou monte o Google Drive. Os arquivos finais devem estar em `data/spider/`.

In [ ]:
# Opção A: você já fez upload do zip
# !python scripts/download_data.py --zip_path /content/spider_data.zip

# Opção B: montar o Google Drive onde o zip está salvo
# from google.colab import drive
# drive.mount('/content/drive')
# !python scripts/download_data.py --zip_path /content/drive/MyDrive/spider_data.zip

# Verificação: todos os caminhos devem existir
from scripts.config import SPIDER_TRAIN_FILE, SPIDER_DEV_FILE, SPIDER_TABLES_FILE, SPIDER_DB_DIR
for p in [SPIDER_TRAIN_FILE, SPIDER_DEV_FILE, SPIDER_TABLES_FILE, SPIDER_DB_DIR]:
    status = "OK" if os.path.exists(p) else "FALTANDO"
    print(f"{status:8s}  {p}")

## 3. Pré-processamento — Spider train → formato chat (Fase 3)

In [ ]:
!python scripts/preprocess_spider.py

## 4. Baseline — Text-to-SQL e MMLU do modelo base (Fase 2 e Fase 5)

In [ ]:
# Avaliação Text-to-SQL do modelo base (Fase 2)
# Use --limit 200 para um teste rápido; remova para o dev split completo (1034 exemplos)
!python scripts/evaluate_text2sql.py --run_name baseline

In [ ]:
# Avaliação MMLU 5-shot do modelo base (Fase 5)
!python scripts/evaluate_mmlu.py --run_name baseline

## 5. Fine-Tuning — duas configurações de hiperparâmetros (Fase 3)

| Config | Épocas | Learning rate | Descrição |
|--------|--------|---------------|-----------|
| A      | 1      | 2e-4          | Treinamento rápido, LR maior |
| B      | 3      | 5e-5          | Mais épocas, LR menor |

Ambas usam QLoRA 4-bit, r=16, α=32, módulos `q_proj k_proj v_proj o_proj`.

In [ ]:
!python scripts/train.py --config configs/config_a.yaml --output_dir adapters/config_a

In [ ]:
!python scripts/train.py --config configs/config_b.yaml --output_dir adapters/config_b

## 6. Avaliação dos modelos fine-tuned (Fases 4 e 5)

In [ ]:
# Text-to-SQL (Fase 4)
!python scripts/evaluate_text2sql.py --adapter_path adapters/config_a --run_name finetuned_a
!python scripts/evaluate_text2sql.py --adapter_path adapters/config_b --run_name finetuned_b

In [ ]:
# MMLU 5-shot (Fase 5)
!python scripts/evaluate_mmlu.py --adapter_path adapters/config_a --run_name finetuned_a
!python scripts/evaluate_mmlu.py --adapter_path adapters/config_b --run_name finetuned_b

## 7. Gate de avaliação via pytest (Fase 4.1)

In [ ]:
import os
os.environ["EVAL_RUN_NAME"] = "finetuned_a"
!python -m pytest scripts/test_evaluation.py -v

## 8. Análise de regressão de capacidade (Fase 5.3)

In [ ]:
!python scripts/analyze_regression.py --baseline_run baseline --finetuned_runs finetuned_a finetuned_b

## 9. Exibição dos resultados consolidados

In [ ]:
import json

with open("results/summary.json") as f:
    summary = json.load(f)

baseline = summary["baseline"]
print("=" * 60)
print(f"{'MODELO':<25} {'Exec. Acc':>10} {'MMLU':>10}")
print("-" * 60)
print(f"{'Baseline':<25} {baseline['execution_accuracy']:>10.4f} {baseline['mmlu_overall']:>10.4f}")

for run, entry in summary["finetuned"].items():
    delta_sql = entry["execution_accuracy_delta_pct"]
    delta_mmlu = entry["mmlu_overall_delta_pct"]
    label = f"Fine-tuned ({run})"
    print(
        f"{label:<25} {entry['execution_accuracy']:>10.4f} {entry['mmlu_overall']:>10.4f}  "
        f"| SQL Δ {delta_sql:+.1f}%  MMLU Δ {delta_mmlu:+.1f}%"
    )

print("=" * 60)
print()
print("MMLU por categoria:")
cats = list(baseline["mmlu_per_category"].keys())
header = f"{'Categoria':<20}" + "".join(f" {'Baseline':>10}") + "".join(
    f" {run:>14}" for run in summary["finetuned"]
)
print(header)
for cat in cats:
    row = f"{cat:<20} {baseline['mmlu_per_category'][cat]:>10.4f}"
    for run, entry in summary["finetuned"].items():
        v = entry["mmlu_per_category"][cat]["finetuned"]
        d = entry["mmlu_per_category"][cat]["delta_pct"]
        row += f"  {v:.4f} ({d:+.1f}%)"
    print(row)

## 10. Análise de erros — exemplos de falha (para o relatório)

In [ ]:
!python scripts/error_analysis.py --run_name finetuned_a --n 3

In [ ]:
# Exibir exemplos de falha de forma estruturada para o relatório
import json

with open("results/error_analysis_finetuned_a.json") as f:
    ea = json.load(f)

print(f"Total de falhas: {ea['total_failures']} / {ea['total_examples']}")
print(f"Execution Accuracy: {ea['execution_accuracy']:.4f}")
print()

for i, rec in enumerate(ea["selected_failures"], 1):
    print(f"─── Falha #{i} ({'db: ' + rec['db_id']}) ───")
    print(f"Pergunta  : {rec['question']}")
    print(f"SQL Gold  : {rec['gold_sql']}")
    print(f"SQL Pred  : {rec['predicted_sql']}")
    print(f"Motivo    : {rec['reason']}")
    print()